In [3]:
import gc
import pickle

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Baseline Training

## Load processed data

In [4]:
DATA_DIR = '/kaggle/input/datasets/f0rzah/artifacts/'

train_final = pd.read_parquet(f'{DATA_DIR}train_final.parquet')
items_features = pd.read_parquet(f'{DATA_DIR}items_features.parquet')
users_features = pd.read_parquet(f'{DATA_DIR}users_features.parquet')

with open(f'{DATA_DIR}evaluation_samples.pkl', 'rb') as f:
    eval_data = pickle.load(f)

print(f'train_final: {train_final.shape}')

train_final: (9178632, 25)


## Feature schema

In [5]:
CAT_FEATURES = ['content_type', 'release_decade', 'age', 'sex', 'income']
DROP_COLS = ['user_id', 'item_id', 'target', 'watched_pct', 'day_of_week']
FEATURES = [c for c in train_final.columns if c not in DROP_COLS]

print(f'Features ({len(FEATURES)})')

Features (20)


## Train / internal-val split

In [6]:
X = train_final[FEATURES]
y = train_final['target']

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

del X, y
gc.collect()

30

## Train CatBoost

In [7]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='Logloss',
    random_seed=42,
    verbose=100,
    cat_features=CAT_FEATURES,
    task_type='GPU',
    devices='0',
    early_stopping_rounds=50,
    use_best_model=True,
)

model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

0:	learn: 0.6164359	test: 0.6164259	best: 0.6164259 (0)	total: 790ms	remaining: 6m 34s
100:	learn: 0.1792665	test: 0.1793468	best: 0.1793468 (100)	total: 1m 9s	remaining: 4m 34s
200:	learn: 0.1775691	test: 0.1777252	best: 0.1777252 (200)	total: 2m 17s	remaining: 3m 25s
300:	learn: 0.1769926	test: 0.1772076	best: 0.1772076 (300)	total: 3m 22s	remaining: 2m 14s
400:	learn: 0.1766386	test: 0.1768970	best: 0.1768970 (400)	total: 4m 27s	remaining: 1m 6s
499:	learn: 0.1764009	test: 0.1766987	best: 0.1766987 (499)	total: 5m 32s	remaining: 0us
bestTest = 0.1766987164
bestIteration = 499


CatBoostClassifier(cat_features=['content_type', 'release_decade', 'age', 'sex', 'income'], depth=6, devices='0', early_stopping_rounds=50, eval_metric='Logloss', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', use_best_model=True, verbose=100)

In [8]:
del X_tr, X_val, y_tr, y_val
gc.collect()

print(f'Best iteration: {model.get_best_iteration()}')
print(f'Best Logloss: {model.get_best_score()["validation"]["Logloss"]:.4f}')

Best iteration: 499
Best Logloss: 0.1767


## Metrics

In [9]:
def compute_metrics(top_k_items, true_items, k=10):
    true_set = set(true_items)
    hits = 0
    sum_precision = 0.0
    sum_ndcg = 0.0
    for rank, item in enumerate(top_k_items[:k], start=1):
        if item in true_set:
            hits += 1
            sum_precision += hits / rank
            sum_ndcg += 1.0 / np.log2(rank + 1)
    num_relevant = min(k, len(true_set))
    if num_relevant == 0:
        return 0.0, 0.0, 0.0
    map_ = sum_precision / num_relevant
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, num_relevant + 1))
    ndcg = sum_ndcg / ideal_dcg if ideal_dcg > 0 else 0.0
    hr = 1.0 if hits > 0 else 0.0
    return hr, ndcg, map_

In [10]:
def evaluate(sampled_users, candidates, ground_truth, batch_size=20, k=10):
    hr_list, ndcg_list, map_list = [], [], []
    for i in tqdm(range(0, len(sampled_users), batch_size)):
        batch = sampled_users[i:i + batch_size]
        rows = [{'user_id': int(u), 'item_id': iid} for u in batch for iid in candidates[int(u)]]
        batch_df = pd.DataFrame(rows)
        batch_df = batch_df.merge(items_features, on='item_id', how='left')
        batch_df = batch_df.merge(users_features, on='user_id', how='left')
        batch_df['is_kids_user_and_kids_content'] = ((batch_df['kids_flg'] == 1) & (batch_df['age_rating_ord'] <= 1)).astype('int8')
        for col in CAT_FEATURES:
            batch_df[col] = batch_df[col].astype(str)

        batch_df['score'] = model.predict_proba(batch_df[FEATURES])[:, 1]

        for user_id in batch:
            user_id = int(user_id)
            preds = batch_df[batch_df['user_id'] == user_id].sort_values('score', ascending=False)
            hr, ndcg, map_ = compute_metrics(preds['item_id'].head(k).tolist(), ground_truth.get(user_id, []), k=k)
            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(map_)
    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))

## Evaluate on train sample

In [11]:
hr_train, ndcg_train, map_train = evaluate(
    eval_data['train_sampled_users'],
    eval_data['train_candidates'],
    eval_data['train_ground_truth'],
)

100%|██████████| 25/25 [00:03<00:00,  7.13it/s]


## Evaluate on val sample

In [12]:
hr_val, ndcg_val, map_val = evaluate(
    eval_data['val_sampled_users'],
    eval_data['val_candidates'],
    eval_data['val_ground_truth'],
)

100%|██████████| 25/25 [00:03<00:00,  7.20it/s]


## Results

In [13]:
results = pd.DataFrame({
    'Metric': ['HR@10', 'NDCG@10', 'MAP@10'],
    'Train': [hr_train, ndcg_train, map_train],
    'Val': [hr_val, ndcg_val, map_val],
})
results['Gap'] = results['Train'] - results['Val']
print(results.to_string(index=False))

 Metric    Train      Val       Gap
  HR@10 0.426000 0.456000 -0.030000
NDCG@10 0.232443 0.217647  0.014797
 MAP@10 0.176882 0.161593  0.015289


In [18]:
importance = model.get_feature_importance()
feat_imp = pd.DataFrame({
    'feature': FEATURES,
    'importance': importance,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(feat_imp.to_string(index=False))

                      feature  importance
          item_popularity_log   59.955508
        item_avg_duration_log   16.461207
      user_total_interactions    6.961608
  user_total_interactions_log    6.036360
            user_avg_duration    3.200062
        user_avg_duration_log    2.816855
                     kids_flg    1.883671
               age_rating_ord    0.826464
                          sex    0.489118
                 content_type    0.441198
                       income    0.298494
                  genre_score    0.167620
                          age    0.163260
                country_score    0.132495
               release_decade    0.068600
            is_recent_release    0.041719
               director_score    0.018094
                is_kids_genre    0.016434
                  actor_score    0.012237
is_kids_user_and_kids_content    0.008995


## Save model

In [19]:
OUTPUT_DIR = '/kaggle/working/'
MODEL_PATH = f'{OUTPUT_DIR}catboost_model.cbm'
model.save_model(MODEL_PATH)

_check = CatBoostClassifier()
_check.load_model(MODEL_PATH)
assert _check.feature_names_ == model.feature_names_
del _check
gc.collect()

print(f'Model saved to {MODEL_PATH}')

Model saved to /kaggle/working/catboost_model.cbm
